In [1]:
import os
import sys
sys.path.append("../../")
from callmefair.util.fair_util import BMInterface
from callmefair.mitigation.fair_grid import BMGridSearch
from callmefair.mitigation.fair_bm import BMType
import pandas as pd
import time
T0 = time.time()  # only grid-search logs written after this point are reported at the end

# (the original `import mkl; mkl.set_num_threads(5)` is Intel-only and was removed)
N_THREADS = min(8, os.cpu_count() or 1)

df = pd.read_csv('../data/wids_pre/wids.csv')

# Defining the name of the label column
label_name = 'DiagPeriodL90D'
# Defininf the name of the privileged group
sensitive_attribute = ['patient_age']

bmI = BMInterface(df, label_name, sensitive_attribute)

privileged_group = [{'patient_age': 1}]
unprivileged_group = [{'patient_age': 0}]

pip install 'aif360[Reductions]'


pip install 'aif360[Reductions]'


pip install 'aif360[Reductions]'


In [2]:
from xgboost import XGBClassifier
from sklearn.svm import LinearSVC  
from sklearn.linear_model import LogisticRegression
from pytorch_tabnet.tab_model import TabNetClassifier
from catboost import CatBoostClassifier
from sklearn.neural_network import MLPClassifier

mlp= MLPClassifier(
    max_iter=500,
    hidden_layer_sizes = (200,200,200,200),
    activation = 'logistic',
    solver = 'adam',
    alpha = 0.01,
    learning_rate = 'adaptive'
)

# Create a new instance of the classifier
lr = LogisticRegression(solver='liblinear')

xgbr =  XGBClassifier(
    max_depth=8,
    learning_rate=0.01,
    gamma = 0.25,
    n_estimators = 500,
    subsample = 0.8,
    colsample_bytree = 0.3)

tabnet = TabNetClassifier()

cat = CatBoostClassifier(eval_metric='Accuracy',
                         depth =  4,
                         learning_rate = 0.01,
                         iterations = 10)


model_list = [tabnet, xgbr, cat, mlp, lr]

In [3]:
tabnet.__str__()#.startswith('XGBClassifier')

"TabNetClassifier(n_d=8, n_a=8, n_steps=3, gamma=1.3, cat_idxs=[], cat_dims=[], cat_emb_dim=[], n_independent=2, n_shared=2, epsilon=1e-15, momentum=0.02, lambda_sparse=0.001, seed=0, clip_value=1, verbose=1, optimizer_fn=<class 'torch.optim.adam.Adam'>, optimizer_params={'lr': 0.02}, scheduler_fn=None, scheduler_params={}, mask_type='sparsemax', input_dim=None, output_dim=None, device_name='auto', n_shared_decoder=1, n_indep_decoder=1, grouped_features=[])"

In [4]:
from itertools import product
from more_itertools import powerset

pre_bm = [BMType.preReweighing,BMType.preDisparate]
pos_bm = [BMType.posCalibrated, BMType.posEqqOds, BMType.posROC]

bm_list = [[pre, pos] for pre, pos in product(pre_bm, pos_bm)]
bm_list.extend([p] for p in pre_bm); bm_list.extend([p] for p in pos_bm)
bm_list

[[<BMType.preReweighing: 1>, <BMType.posCalibrated: 6>],
 [<BMType.preReweighing: 1>, <BMType.posEqqOds: 7>],
 [<BMType.preReweighing: 1>, <BMType.posROC: 8>],
 [<BMType.preDisparate: 2>, <BMType.posCalibrated: 6>],
 [<BMType.preDisparate: 2>, <BMType.posEqqOds: 7>],
 [<BMType.preDisparate: 2>, <BMType.posROC: 8>],
 [<BMType.preReweighing: 1>],
 [<BMType.preDisparate: 2>],
 [<BMType.posCalibrated: 6>],
 [<BMType.posEqqOds: 7>],
 [<BMType.posROC: 8>]]

In [5]:
import copy

bm_list_in = copy.deepcopy(bm_list)
[l.append(BMType.inAdversarial) for l in bm_list_in]
bm_list_in.extend(bm_list)
bm_list_in.append([BMType.inAdversarial])
bm_list_in

[[<BMType.preReweighing: 1>,
  <BMType.posCalibrated: 6>,
  <BMType.inAdversarial: 4>],
 [<BMType.preReweighing: 1>, <BMType.posEqqOds: 7>, <BMType.inAdversarial: 4>],
 [<BMType.preReweighing: 1>, <BMType.posROC: 8>, <BMType.inAdversarial: 4>],
 [<BMType.preDisparate: 2>,
  <BMType.posCalibrated: 6>,
  <BMType.inAdversarial: 4>],
 [<BMType.preDisparate: 2>, <BMType.posEqqOds: 7>, <BMType.inAdversarial: 4>],
 [<BMType.preDisparate: 2>, <BMType.posROC: 8>, <BMType.inAdversarial: 4>],
 [<BMType.preReweighing: 1>, <BMType.inAdversarial: 4>],
 [<BMType.preDisparate: 2>, <BMType.inAdversarial: 4>],
 [<BMType.posCalibrated: 6>, <BMType.inAdversarial: 4>],
 [<BMType.posEqqOds: 7>, <BMType.inAdversarial: 4>],
 [<BMType.posROC: 8>, <BMType.inAdversarial: 4>],
 [<BMType.preReweighing: 1>, <BMType.posCalibrated: 6>],
 [<BMType.preReweighing: 1>, <BMType.posEqqOds: 7>],
 [<BMType.preReweighing: 1>, <BMType.posROC: 8>],
 [<BMType.preDisparate: 2>, <BMType.posCalibrated: 6>],
 [<BMType.preDisparate: 

In [6]:
for model in model_list:
    bmG = BMGridSearch(bmI, model=model, bm_list=bm_list, privileged_group=privileged_group, unprivileged_group=unprivileged_group)
    bmG.run_single_sensitive()

epoch 0  | loss: 0.66987 | val_0_auc: 0.7171  |  0:00:00s


epoch 1  | loss: 0.6114  | val_0_auc: 0.75032 |  0:00:00s


epoch 2  | loss: 0.54285 | val_0_auc: 0.75395 |  0:00:00s


epoch 3  | loss: 0.50212 | val_0_auc: 0.78006 |  0:00:01s


epoch 4  | loss: 0.49356 | val_0_auc: 0.76958 |  0:00:01s


epoch 5  | loss: 0.48733 | val_0_auc: 0.77362 |  0:00:01s


... (312 outputs left out)

0:	learn: 0.8132007	total: 2.38ms	remaining: 21.4ms
1:	learn: 0.8132007	total: 3.74ms	remaining: 14.9ms
2:	learn: 0.8132007	total: 5.04ms	remaining: 11.8ms
3:	learn: 0.8131032	total: 6.32ms	remaining: 9.48ms
4:	learn: 0.8132007	total: 7.65ms	remaining: 7.65ms
5:	learn: 0.8132007	total: 8.95ms	remaining: 5.97ms
6:	learn: 0.8132007	total: 10.4ms	remaining: 4.47ms
7:	learn: 0.8132007	total: 11.8ms	remaining: 2.95ms
8:	learn: 0.8132007	total: 13.1ms	remaining: 1.46ms
9:	learn: 0.8132007	total: 14.4ms	remaining: 0us


0:	learn: 0.8132007	total: 1.2ms	remaining: 10.8ms
1:	learn: 0.8132007	total: 2.78ms	remaining: 11.1ms
2:	learn: 0.8132007	total: 4.07ms	remaining: 9.49ms
3:	learn: 0.8131032	total: 5.54ms	remaining: 8.31ms
4:	learn: 0.8132007	total: 6.68ms	remaining: 6.68ms
5:	learn: 0.8132007	total: 8.02ms	remaining: 5.35ms
6:	learn: 0.8132007	total: 9.33ms	remaining: 4ms
7:	learn: 0.8132007	total: 10.7ms	remaining: 2.67ms
8:	learn: 0.8132007	total: 11.9ms	remaining: 1.33ms
9:	learn: 0.8132007	total: 13.4ms	remaining: 0us


0:	learn: 0.8130884	total: 1.2ms	remaining: 10.8ms
1:	learn: 0.8130884	total: 2.76ms	remaining: 11ms
2:	learn: 0.8130884	total: 4.23ms	remaining: 9.87ms
3:	learn: 0.8130884	total: 5.5ms	remaining: 8.25ms
4:	learn: 0.8130884	total: 6.72ms	remaining: 6.72ms
5:	learn: 0.8130884	total: 8.01ms	remaining: 5.34ms
6:	learn: 0.8130884	total: 9.14ms	remaining: 3.92ms
7:	learn: 0.8130884	total: 10.5ms	remaining: 2.61ms
8:	learn: 0.8130884	total: 11.9ms	remaining: 1.32ms
9:	learn: 0.8130884	total: 13.3ms	remaining: 0us


0:	learn: 0.8132007	total: 1.19ms	remaining: 10.7ms
1:	learn: 0.8132007	total: 2.58ms	remaining: 10.3ms
2:	learn: 0.8132007	total: 3.89ms	remaining: 9.09ms
3:	learn: 0.8131032	total: 5.21ms	remaining: 7.81ms
4:	learn: 0.8132007	total: 6.5ms	remaining: 6.5ms
5:	learn: 0.8132007	total: 7.95ms	remaining: 5.3ms
6:	learn: 0.8132007	total: 9.27ms	remaining: 3.97ms
7:	learn: 0.8132007	total: 10.7ms	remaining: 2.67ms
8:	learn: 0.8132007	total: 12.1ms	remaining: 1.34ms
9:	learn: 0.8132007	total: 13.6ms	remaining: 0us
0:	learn: 0.8132007	total: 1.13ms	remaining: 10.1ms
1:	learn: 0.8132007	total: 2.38ms	remaining: 9.52ms
2:	learn: 0.8132007	total: 3.61ms	remaining: 8.43ms
3:	learn: 0.8132007	total: 5.17ms	remaining: 7.76ms
4:	learn: 0.8132007	total: 6.56ms	remaining: 6.56ms
5:	learn: 0.8132007	total: 7.85ms	remaining: 5.23ms
6:	learn: 0.8132007	total: 9.43ms	remaining: 4.04ms
7:	learn: 0.8132007	total: 10.7ms	remaining: 2.69ms
8:	learn: 0.8132007	total: 12ms	remaining: 1.34ms
9:	learn: 0.8132007	

In [7]:
bmG = BMGridSearch(bmI, model=None, bm_list=[[BMType.inAdversarial]], privileged_group=privileged_group, unprivileged_group=unprivileged_group)
bmG.run_single_sensitive()

Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.


Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.


epoch 0; iter: 0; batch classifier loss: 0.698153
epoch 1; iter: 0; batch classifier loss: 0.439028
epoch 2; iter: 0; batch classifier loss: 0.419618
epoch 3; iter: 0; batch classifier loss: 0.484637


2026-11-05 18:35:59.934792: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization pass is not enabled


epoch 4; iter: 0; batch classifier loss: 0.448866


epoch 5; iter: 0; batch classifier loss: 0.380953
epoch 6; iter: 0; batch classifier loss: 0.454328
epoch 7; iter: 0; batch classifier loss: 0.439650
epoch 8; iter: 0; batch classifier loss: 0.502509
epoch 9; iter: 0; batch classifier loss: 0.369921


... (24 outputs left out)

epoch 40; iter: 0; batch classifier loss: 0.903440; batch adversarial loss: 0.149613
epoch 41; iter: 0; batch classifier loss: 0.857391; batch adversarial loss: 0.092119
epoch 42; iter: 0; batch classifier loss: 0.689689; batch adversarial loss: 0.032585
epoch 43; iter: 0; batch classifier loss: 0.596018; batch adversarial loss: 0.029515


epoch 44; iter: 0; batch classifier loss: 0.590034; batch adversarial loss: 0.028193
epoch 45; iter: 0; batch classifier loss: 0.626805; batch adversarial loss: 0.026239
epoch 46; iter: 0; batch classifier loss: 0.538291; batch adversarial loss: 0.024396
epoch 47; iter: 0; batch classifier loss: 0.449555; batch adversarial loss: 0.148724


epoch 48; iter: 0; batch classifier loss: 0.524343; batch adversarial loss: 0.084288
epoch 49; iter: 0; batch classifier loss: 0.483063; batch adversarial loss: 0.085617


[{'model': 'AdversarialDebiasing',
  'BM': 'baseline',
  'balanced_acc': 0.7358359105060583,
  'acc': 0.7918939984411536,
  'precision': 0.7654075546719682,
  'recall': 0.9612983770287141,
  'f1': 0.8522412838959601,
  'mcc': 0.3338110190175954,
  'eq_opp_diff': -0.03904282115869018,
  'avg_odd_diff': -0.27523866629660076,
  'spd': -0.21725490196078434,
  'disparate_impact': 0.7827450980392157,
  'theil_idx': 0.0716374161744098,
  'fair_score': 0.928},
 {'model': 'AdversarialDebiasing',
  'BM': 'inAD',
  'balanced_acc': 0.7381126289233894,
  'acc': 0.7926734216679657,
  'precision': 0.7677677677677678,
  'recall': 0.9575530586766542,
  'f1': 0.8522222222222222,
  'mcc': 0.30126950168893946,
  'eq_opp_diff': -0.04282115869017633,
  'avg_odd_diff': -0.281285839220348,
  'spd': -0.22274509803921572,
  'disparate_impact': 0.7772549019607843,
  'theil_idx': 0.07364782568869843,
  'fair_score': 0.945}]

In [8]:
# The original looped `for bm in bm_list` while ignoring `bm`, re-running the identical
# grid 11 times; a single run is performed here.
bmG = BMGridSearch(bmI, model=None, bm_list=bm_list_in, privileged_group=privileged_group, unprivileged_group=unprivileged_group)
bmG.run_single_sensitive()

epoch 0; iter: 0; batch classifier loss: 0.763538
epoch 1; iter: 0; batch classifier loss: 0.484891
epoch 2; iter: 0; batch classifier loss: 0.478746
epoch 3; iter: 0; batch classifier loss: 0.512645
epoch 4; iter: 0; batch classifier loss: 0.522095


epoch 5; iter: 0; batch classifier loss: 0.415687
epoch 6; iter: 0; batch classifier loss: 0.525761
epoch 7; iter: 0; batch classifier loss: 0.508431
epoch 8; iter: 0; batch classifier loss: 0.410698
epoch 9; iter: 0; batch classifier loss: 0.419920
epoch 10; iter: 0; batch classifier loss: 0.547273


epoch 11; iter: 0; batch classifier loss: 0.446046
epoch 12; iter: 0; batch classifier loss: 0.495060
epoch 13; iter: 0; batch classifier loss: 0.521221
epoch 14; iter: 0; batch classifier loss: 0.375454
epoch 15; iter: 0; batch classifier loss: 0.503395
epoch 16; iter: 0; batch classifier loss: 0.499073


epoch 17; iter: 0; batch classifier loss: 0.469693
epoch 18; iter: 0; batch classifier loss: 0.582192
epoch 19; iter: 0; batch classifier loss: 0.497301
epoch 20; iter: 0; batch classifier loss: 0.514089
epoch 21; iter: 0; batch classifier loss: 0.550867
epoch 22; iter: 0; batch classifier loss: 0.473237


epoch 23; iter: 0; batch classifier loss: 0.444184
epoch 24; iter: 0; batch classifier loss: 0.408883
epoch 25; iter: 0; batch classifier loss: 0.452559
epoch 26; iter: 0; batch classifier loss: 0.421012
epoch 27; iter: 0; batch classifier loss: 0.480466
epoch 28; iter: 0; batch classifier loss: 0.545077


epoch 29; iter: 0; batch classifier loss: 0.547490
epoch 30; iter: 0; batch classifier loss: 0.402554
epoch 31; iter: 0; batch classifier loss: 0.503372
epoch 32; iter: 0; batch classifier loss: 0.462120
epoch 33; iter: 0; batch classifier loss: 0.391765
epoch 34; iter: 0; batch classifier loss: 0.410686


... (255 outputs left out)

epoch 40; iter: 0; batch classifier loss: 0.525863; batch adversarial loss: 0.089459
epoch 41; iter: 0; batch classifier loss: 0.421690; batch adversarial loss: 0.091111
epoch 42; iter: 0; batch classifier loss: 0.607262; batch adversarial loss: 0.018580
epoch 43; iter: 0; batch classifier loss: 0.478985; batch adversarial loss: 0.019833


epoch 44; iter: 0; batch classifier loss: 0.524538; batch adversarial loss: 0.020013
epoch 45; iter: 0; batch classifier loss: 0.609011; batch adversarial loss: 0.093846
epoch 46; iter: 0; batch classifier loss: 0.466316; batch adversarial loss: 0.018186
epoch 47; iter: 0; batch classifier loss: 0.509526; batch adversarial loss: 0.016674


epoch 48; iter: 0; batch classifier loss: 0.538087; batch adversarial loss: 0.014631
epoch 49; iter: 0; batch classifier loss: 0.392513; batch adversarial loss: 0.086843


[{'model': 'AdversarialDebiasing',
  'BM': 'baseline',
  'balanced_acc': 0.739361068374076,
  'acc': 0.79423226812159,
  'precision': 0.7682317682317682,
  'recall': 0.9600499375780275,
  'f1': 0.853496115427303,
  'mcc': 0.32186633843148876,
  'eq_opp_diff': -0.04030226700251893,
  'avg_odd_diff': -0.2800263933765193,
  'spd': -0.2211764705882353,
  'disparate_impact': 0.7788235294117647,
  'theil_idx': 0.07200542624022259,
  'fair_score': 0.939},
 {'model': 'AdversarialDebiasing',
  'BM': 'preReweighing inAD posCalibrated',
  'balanced_acc': 0.5,
  'acc': 0.6243180046765393,
  'precision': 0.6243180046765393,
  'recall': 1.0,
  'f1': 0.7687140115163147,
  'mcc': 0.002477154093548269,
  'eq_opp_diff': 0.0,
  'avg_odd_diff': 0.0,
  'spd': 0.0,
  'disparate_impact': 1.0,
  'theil_idx': 0.05963049839826032,
  'fair_score': 0.021},
 {'model': 'AdversarialDebiasing',
  'BM': 'preReweighing inAD posEqqOds',
  'balanced_acc': 0.5,
  'acc': 0.37568199532346064,
  'precision': 0.0,
  'recall':

In [9]:
# Show what the grid searches logged (one CSV per run_single_sensitive call under ./results)
import glob
frames = [pd.read_csv(f).assign(source=os.path.basename(f)) for f in [f for f in sorted(glob.glob('results/experiment_*.csv'), key=os.path.getmtime) if os.path.getmtime(f) >= T0]]
logged = pd.concat(frames, ignore_index=True)
logged[['model','BM','balanced_acc','acc','spd','fair_score','source']]

,model,BM,balanced_acc,acc,spd,fair_score,source
0,TabNetClassifier,baseline,0.737286,0.792673,-0.219608,0.935,experiment_20261105_183326_394201.csv
1,TabNetClassifier,preReweighing posCalibrated,0.686677,0.756820,-0.172549,0.700,experiment_20261105_183326_394201.csv
2,TabNetClassifier,preReweighing posEqqOds,0.622107,0.664848,-0.299608,1.000,experiment_20261105_183326_394201.csv
3,TabNetClassifier,preReweighing posROC,0.737286,0.792673,-0.219608,0.935,experiment_20261105_183326_394201.csv
4,TabNetClassifier,preDisparate posCalibrated,0.681077,0.752923,-0.167059,0.685,experiment_20261105_183326_394201.csv
...,...,...,...,...,...,...,...
81,AdversarialDebiasing,preDisparate,0.738324,0.793453,-0.220392,0.937,experiment_20261105_183724_754976.csv
82,AdversarialDebiasing,posCalibrated,0.642080,0.724864,-0.132549,0.595,experiment_20261105_183724_754976.csv
83,AdversarialDebiasing,posEqqOds,0.708874,0.773188,-0.190588,0.746,experiment_20261105_183724_754976.csv
84,AdversarialDebiasing,posROC,0.739361,0.794232,-0.221176,0.939,experiment_20261105_183724_754976.csv
